In [8]:
import sys
import os
import xarray as xr
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pyproj import Transformer
from scipy import interpolate
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import earthaccess
from geopy.geocoders import Nominatim
from datetime import datetime, timedelta
import pandas as pd

In [9]:
# Use these lines if your util folder is in the src directory
project_src_path = "/home/jovyan/Desktop/swot-surf/src"
if project_src_path not in sys.path:
    sys.path.append(project_src_path)
import util.plotting_helpers as plothelp

auth = earthaccess.login(strategy="netrc")

class SWOTAnalyzer:
    """
    A class to perform analysis and visualization of SWOT L2 Water Mask data.
    """
    def __init__(self, bounding_box, temporal_range=None, short_name="SWOT_L2_HR_Raster_D"):
        """
        Initializes the analyzer with a bounding box and an optional temporal range.
        The short_name parameter is now configurable.
        """
        self.short_name = short_name
        self.bounding_box = bounding_box
        self.temporal_range = temporal_range if temporal_range else ('2023-01-01', '2025-12-31')
        self.all_granules = []
        self.granule_info = {}
        self.swot_datasets = []

        print(f"\n--- Initializing for bounding box: {self.bounding_box} ---")
        self._get_available_granules()
        if not self.granule_info:
            print("No valid granules found for the specified bounding box. Please check coordinates.")

    def _get_available_granules(self):
        """
        Searches Earthdata for all SWOT granules within the initialized bounding box
        and populates self.granule_info.
        """
        print(f"Searching for {self.short_name} granules in the specified area and timeframe...")
        try:
            self.all_granules = earthaccess.search_data(
                short_name=self.short_name,
                bounding_box=self.bounding_box,
                temporal=self.temporal_range
            )
            print(f"Found {len(self.all_granules)} granules in total.")

            if not self.all_granules:
                return

            # --- NEW DIAGNOSTIC PRINT ---
            print("\n--- Inspecting the first granule object for available attributes ---")
            first_granule = self.all_granules[0]
            print(f"Attributes for a '{self.short_name}' granule:")
            print(dir(first_granule))
            print("\n---------------------------------------------------\n")
            # --- END DIAGNOSTIC ---

            # The logic below assumes a specific metadata structure.
            # You may need to update the key names based on the diagnostic output.
            for granule in self.all_granules:
                granule_id = granule.get('title')
                granule_date = granule.get('data_start')
                
                if granule_id and granule_date:
                    self.granule_info[granule_id] = {
                        'date': granule_date[:10],
                        'granule_obj': granule
                    }
        except Exception as e:
            print(f"Error during Earth Data search: {e}")
            self.all_granules = []
            self.granule_info = {}
    
    def _prompt_and_select_granule_pair(self):
        if not self.granule_info:
            print("No granules available for selection. Exiting.")
            return None, None

        print("\n--- Available SWOT Granules (ID and Date) ---")
        sorted_granule_ids = sorted(self.granule_info.keys(), key=lambda k: self.granule_info[k]['date'])
        
        for i, granule_id in enumerate(sorted_granule_ids):
            info = self.granule_info[granule_id]
            print(f"  {i+1}: ID: {granule_id}, Date: {info['date']}")

        while True:
            before_input = input("\nEnter the ID of the 'BEFORE' granule you want to analyze: ").strip()
            if before_input in self.granule_info:
                before_granule_obj = self.granule_info[before_input]['granule_obj']
                break
            else:
                print("Invalid ID. Please enter an ID from the list above.")
        
        while True:
            after_input = input("Enter the ID of the 'AFTER' granule you want to analyze: ").strip()
            if after_input in self.granule_info:
                after_granule_obj = self.granule_info[after_input]['granule_obj']
                break
            else:
                print("Invalid ID. Please enter an ID from the list above.")

        print(f"\nDownloading and opening 'BEFORE' granule: {before_input}...")
        try:
            ds_before = xr.open_dataset(earthaccess.open([before_granule_obj])[0], engine="h5netcdf")
        except Exception as e:
            print(f"Error opening 'BEFORE' granule {before_input}: {e}")
            ds_before = None

        print(f"Downloading and opening 'AFTER' granule: {after_input}...")
        try:
            ds_after = xr.open_dataset(earthaccess.open([after_granule_obj])[0], engine="h5netcdf")
        except Exception as e:
            print(f"Error opening 'AFTER' granule {after_input}: {e}")
            ds_after = None
        
        return ds_before, ds_after

    def _plot_swot_data(self, ax, ds, title, vmin, vmax):
        if ds is None:
            ax.set_title(f"{title}\n(No data available)", fontsize=14)
            ax.set_visible(False)
            return None, None
        wse = ds["wse"] + ds["height_cor_xover"]
        utm_zone = ds.utm_zone_num
        utm_crs = ccrs.UTM(zone=utm_zone, southern_hemisphere=False)
        transformer = Transformer.from_crs(utm_crs, ccrs.PlateCarree(), always_xy=True)
        x_utm, y_utm = wse["x"].values, wse["y"].values
        X_utm, Y_utm = np.meshgrid(x_utm, y_utm)
        X_lon, Y_lat = transformer.transform(X_utm, Y_utm)
        mesh = ax.pcolormesh(
            X_lon, Y_lat, wse.values, transform=ccrs.PlateCarree(), cmap="viridis", vmin=vmin, vmax=vmax,
        )
        ax.gridlines(draw_labels=True)
        ax.add_feature(cfeature.LAND, facecolor='lightgray')
        ax.add_feature(cfeature.OCEAN, facecolor='lightblue')
        ax.coastlines(resolution='10m', linewidth=1)
        ax.set_title(title, fontsize=14)
        return mesh, wse

    def _plot_difference(self, ax, ds_before, ds_after, title, vmin, vmax):
        if ds_before is None or ds_after is None:
            ax.set_title(f"{title}\n(Cannot compute difference due to missing data)", fontsize=14)
            ax.set_visible(False)
            return None
        wse_before = ds_before["wse"] + ds_before["height_cor_xover"]
        wse_after = ds_after["wse"] + ds_after["height_cor_xover"]
        if ds_before.utm_zone_num != ds_after.utm_zone_num:
            print(f"Warning: UTM zones differ. Reprojecting 'before' to 'after's CRS for difference calculation.")
            transformer_before_to_latlon = Transformer.from_crs(
                ccrs.UTM(zone=ds_before.utm_zone_num, southern_hemisphere=False), ccrs.PlateCarree(), always_xy=True)
            transformer_after_to_latlon = Transformer.from_crs(
                ccrs.UTM(zone=ds_after.utm_zone_num, southern_hemisphere=False), ccrs.PlateCarree(), always_xy=True)
            x_before_utm, y_before_utm = wse_before["x"].values, wse_before["y"].values
            X_before_utm, Y_before_utm = np.meshgrid(x_before_utm, y_before_utm)
            lon_before, lat_before = transformer_before_to_latlon.transform(X_before_utm, Y_before_utm)
            x_after_utm, y_after_utm = wse_after["x"].values, wse_after["y"].values
            X_after_utm, Y_after_utm = np.meshgrid(x_after_utm, y_after_utm)
            lon_after, lat_after = transformer_after_to_latlon.transform(X_after_utm, Y_after_utm)
            wse_before_ll = xr.DataArray(wse_before.values, coords={'lat': lat_before[:,0], 'lon': lon_before[0,:]}, dims=['y', 'x']).rename({'y': 'lat', 'x': 'lon'})
            wse_after_ll = xr.DataArray(wse_after.values, coords={'lat': lat_after[:,0], 'lon': lon_after[0,:]}, dims=['y', 'x']).rename({'y': 'lat', 'x': 'lon'})
            try:
                wse_before_interpolated = wse_before_ll.interp(lat=wse_after_ll.lat, lon=wse_after_ll.lon, method="linear")
                wse_diff = wse_after_ll - wse_before_interpolated
            except Exception as e:
                print(f"Error during interpolation for difference: {e}.")
                ax.set_title(f"{title}\n(Error in re-gridding)", fontsize=14)
                ax.set_visible(False)
                return None
            X_lon, Y_lat = lon_after, lat_after
        else:
            try:
                wse_before_aligned = wse_before.interp(x=wse_after.x, y=wse_after.y, method="linear")
                wse_diff = wse_after - wse_before_aligned
            except Exception as e:
                print(f"Error interpolating WSE 'before' onto 'after' grid: {e}. Assuming perfect alignment.")
                if wse_before.shape == wse_after.shape and np.allclose(wse_before['x'], wse_after['x']) and np.allclose(wse_before['y'], wse_after['y']):
                    wse_diff = wse_after - wse_before
                else:
                    print("Direct difference not possible due to incompatible grids. Skipping difference plot.")
                    ax.set_title(f"{title}\n(Incompatible data grids)", fontsize=14)
                    ax.set_visible(False)
                    return None
            utm_crs_plot = ccrs.UTM(zone=ds_before.utm_zone_num, southern_hemisphere=False)
            transformer = Transformer.from_crs(utm_crs_plot, ccrs.PlateCarree(), always_xy=True)
            x_utm, y_utm = wse_after["x"].values, wse_after["y"].values
            X_utm, Y_utm = np.meshgrid(x_utm, y_utm)
            X_lon, Y_lat = transformer.transform(X_utm, Y_utm)
        mesh = ax.pcolormesh(
            X_lon, Y_lat, wse_diff.values, transform=ccrs.PlateCarree(), cmap="RdBu", vmin=vmin, vmax=vmax,
        )
        ax.gridlines(draw_labels=True)
        ax.add_feature(cfeature.LAND, facecolor='lightgray')
        ax.add_feature(cfeature.OCEAN, facecolor='lightblue')
        ax.coastlines(resolution='10m', linewidth=1)
        ax.set_title(title, fontsize=14)
        return mesh

    def run_analysis(self):
        num_pairs = int(input("How many SWOT granule pairs (before and after) do you want to analyze? "))
        for i in range(num_pairs):
            print(f"\n--- Selecting granules for Pair {i+1} ---")
            ds_before, ds_after = self._prompt_and_select_granule_pair()
            if ds_before is not None and ds_after is not None:
                self.swot_datasets.append((ds_before, ds_after))
            else:
                print(f"Skipping Pair {i+1} due to unselected or unopenable granules.")
        if not self.swot_datasets:
            print("No valid granule pairs were found or entered. Exiting.")
            return
        num_rows = len(self.swot_datasets)
        num_cols = 3
        fig = plt.figure(figsize=(24, 8 * num_rows))
        gs = fig.add_gridspec(num_rows, num_cols + 2, width_ratios=[1, 1, 1, 0.05, 0.05], wspace=0.3)
        main_location_title = input("Enter a main location title for all plots: ")
        for i, (ds_before, ds_after) in enumerate(self.swot_datasets):
            before_date = ds_before.data_start_date_time[:10] if hasattr(ds_before, 'data_start_date_time') else "N/A"
            after_date = ds_after.data_start_date_time[:10] if hasattr(ds_after, 'data_start_date_time') else "N/A"
            title_before = f"WSE Before ({before_date})"
            title_after = f"WSE After ({after_date})"
            title_diff = f"WSE Difference ({before_date} vs {after_date})"
            ax1 = fig.add_subplot(gs[i, 0], projection=ccrs.PlateCarree())
            mesh1, wse1 = self._plot_swot_data(ax1, ds_before, title_before, vmin=0, vmax=10)
            ax2 = fig.add_subplot(gs[i, 1], projection=ccrs.PlateCarree())
            mesh2, wse2 = self._plot_swot_data(ax2, ds_after, title_after, vmin=0, vmax=10)
            ax3 = fig.add_subplot(gs[i, 2], projection=ccrs.PlateCarree())
            mesh3 = self._plot_difference(ax3, ds_before, ds_after, title_diff, vmin=-5, vmax=5)
            if mesh1:
                cax_wse = fig.add_subplot(gs[i, num_cols])
                cbar_wse = fig.colorbar(mesh1, cax=cax_wse, orientation='vertical')
                cbar_wse.set_label("Water Surface Elevation [m]", fontsize=12, rotation=90)
            if mesh3:
                cax_diff = fig.add_subplot(gs[i, num_cols + 1])
                cbar_diff = fig.colorbar(mesh3, cax=cax_diff, orientation='vertical')
                cbar_diff.set_label("WSE Difference [m]", fontsize=12, rotation=90)
        fig.text(x=0.5, y=0.98, s=main_location_title, fontsize=20, color='black', va='top', ha='center')
        plt.show()

    def _analyze_pre_flood_progression(self, num_granules, flood_date):
        if not self.all_granules:
            print("No granules available for pre-flood analysis.")
            return

        sorted_granules = sorted(
            self.all_granules,
            key=lambda g: g.get('data_start', '9999-99-99T99:99:99Z')
        )
        
        pre_flood_granules = []
        for granule in reversed(sorted_granules):
            if granule.get('data_start') and granule.get('data_start') < flood_date:
                pre_flood_granules.append(granule)
                if len(pre_flood_granules) >= num_granules:
                    break

        if not pre_flood_granules:
            print(f"No granules found in the timeframe leading up to {flood_date}.")
            return

        pre_flood_granules.reverse()
        
        print(f"\nDownloading and opening {len(pre_flood_granules)} granules for analysis...")
        datasets = []
        for granule in pre_flood_granules:
            try:
                ds = xr.open_dataset(earthaccess.open([granule])[0], engine="h5netcdf")
                datasets.append(ds)
            except Exception as e:
                print(f"Error opening granule {granule.get('title')}: {e}")
        
        if not datasets:
            print("No datasets were successfully opened for pre-flood analysis.")
            return
        
        dates = [ds.data_start_date_time[:10] for ds in datasets]
        avg_wses = [np.nanmean(ds['wse'] + ds['height_cor_xover']) for ds in datasets]
        
        plt.figure(figsize=(10, 6))
        plt.plot(pd.to_datetime(dates), avg_wses, marker='o', linestyle='-')
        plt.title(f"Average WSE Progression Before Flooding")
        plt.xlabel("Date")
        plt.ylabel("Average Water Surface Elevation (m)")
        plt.grid(True)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

    def _animate_wse_progression(self, num_granules, flood_date):
        if not self.all_granules:
            print("No granules available for animation.")
            return

        sorted_granules = sorted(
            self.all_granules,
            key=lambda g: g.get('data_start', '9999-99-99T99:99:99Z')
        )
        
        pre_flood_granules = []
        for granule in reversed(sorted_granules):
            if granule.get('data_start') and granule.get('data_start') < flood_date:
                pre_flood_granules.append(granule)
                if len(pre_flood_granules) >= num_granules:
                    break
        
        if not pre_flood_granules:
            print(f"No granules found in the timeframe leading up to {flood_date}.")
            return
        
        pre_flood_granules.reverse()
        
        print(f"\nDownloading and opening {len(pre_flood_granules)} granules for animation...")
        datasets = []
        for granule in pre_flood_granules:
            try:
                ds = xr.open_dataset(earthaccess.open([granule])[0], engine="h5netcdf")
                datasets.append(ds)
            except Exception as e:
                print(f"Error opening granule {granule.get('title')}: {e}")
        
        if not datasets:
            print("No datasets were successfully opened for animation.")
            return

        fig, ax = plt.subplots(figsize=(12, 8), subplot_kw={'projection': ccrs.PlateCarree()})
        fig.suptitle(f"WSE Progression Before Flooding", fontsize=16)

        def update(frame):
            ax.clear()
            ds = datasets[frame]
            title = f"Date: {ds.data_start_date_time[:10]}"
            self._plot_swot_data(ax, ds, title, vmin=0, vmax=10)

        anim = FuncAnimation(fig, update, frames=len(datasets), interval=1000)
        
        print("Saving animation as 'wse_progression.gif'...")
        try:
            anim.save('wse_progression.gif', writer='pillow', fps=1)
            print("Animation saved successfully!")
        except Exception as e:
            print(f"Error saving animation: {e}")
        
        plt.close(fig)

# --- MAIN EXECUTION BLOCK ---
if __name__ == "__main__":
    while True:
        mode = input("Enter mode ('scientist', 'student', or 'flood_progression'): ").lower().strip()
        if mode in ['scientist', 'student', 'flood_progression']:
            break
        print("Invalid mode. Please enter 'scientist', 'student', or 'flood_progression'.")

    # This prompt allows you to choose the SWOT version you want to use
    short_name_input = input("Enter the SWOT data product short name (e.g., 'SWOT_L2_HR_Raster_D'): ").strip()
    
    # SCIENTIST MODE
    if mode == 'scientist':
        print("--- Scientist Mode: Enter bounding box coordinates ---")
        try:
            min_lon = float(input("Minimum longitude (e.g., -92.5): "))
            min_lat = float(input("Minimum latitude (e.g., 30.0): "))
            max_lon = float(input("Maximum longitude (e.g., -90.0): "))
            max_lat = float(input("Maximum latitude (e.g., 32.0): "))
            user_bounding_box = (min_lon, min_lat, max_lon, max_lat)
            analyzer = SWOTAnalyzer(user_bounding_box, short_name=short_name_input)
            analyzer.run_analysis()
        except ValueError:
            print("Invalid input. Please enter numbers for the coordinates.")
            sys.exit(1)
            
    # STUDENT MODE
    elif mode == 'student':
        print("--- Student Mode: Enter a country and a timeframe ---")
        country_name = input("Enter a country name (e.g., India): ").strip()
        start_date_str = input("Enter start date (YYYY-MM-DD): ").strip()
        end_date_str = input("Enter end date (YYYY-MM-DD): ").strip()
        try:
            geolocator = Nominatim(user_agent="swot_analyzer")
            location = geolocator.geocode(country_name)
            if not location:
                print(f"Could not find coordinates for {country_name}. Please try a different name.")
                sys.exit(1)
            bbox = [float(coord) for coord in location.raw['boundingbox']]
            user_bounding_box = (bbox[2], bbox[0], bbox[3], bbox[1])
            start_date = datetime.strptime(start_date_str, '%Y-%m-%d').strftime('%Y-%m-%d')
            end_date = datetime.strptime(end_date_str, '%Y-%m-%d').strftime('%Y-%m-%d')
            temporal_range = (start_date, end_date)
            analyzer = SWOTAnalyzer(user_bounding_box, temporal_range=temporal_range, short_name=short_name_input)
            analyzer.run_analysis()
        except ValueError:
            print("Invalid date format. Please use YYYY-MM-DD.")
            sys.exit(1)
        except Exception as e:
            print(f"An unexpected error occurred: {e}")
            sys.exit(1)

    # FLOOD PROGRESSION MODE
    elif mode == 'flood_progression':
        print("--- Flood Progression Mode: Analyze WSE leading up to a flood event ---")
        try:
            min_lon = float(input("Minimum longitude (e.g., -92.5): "))
            min_lat = float(input("Minimum latitude (e.g., 30.0): "))
            max_lon = float(input("Maximum longitude (e.g., -90.0): "))
            max_lat = float(input("Maximum latitude (e.g., 32.0): "))
            user_bounding_box = (min_lon, min_lat, max_lon, max_lat)

            flood_date_str = input("Enter the flood event date (YYYY-MM-DD): ")
            days_in_advance = int(input("How many days in advance do you want to look? "))
            num_granules = int(input("How many granules do you want to analyze? (e.g., 5): "))
            
            flood_date_obj = datetime.strptime(flood_date_str, '%Y-%m-%d')
            start_date_obj = flood_date_obj - timedelta(days=days_in_advance)
            temporal_range = (start_date_obj.strftime('%Y-%m-%d'), flood_date_obj.strftime('%Y-%m-%d'))
            
            analyzer = SWOTAnalyzer(user_bounding_box, temporal_range=temporal_range, short_name=short_name_input)
            if analyzer.granule_info:
                analyzer._analyze_pre_flood_progression(num_granules, flood_date_str)
                analyzer._animate_wse_progression(num_granules, flood_date_str)
            else:
                print("Cannot perform progression analysis without any granules.")

        except ValueError:
            print("Invalid input. Please enter numbers for coordinates/days, and use YYYY-MM-DD for the date.")
            sys.exit(1)
        except Exception as e:
            print(f"An unexpected error occurred: {e}")
            sys.exit(1)

Enter mode ('scientist', 'student', or 'flood_progression'):  flood_progression
Enter the SWOT data product short name (e.g., 'SWOT_L2_HR_Raster_D'):  SWOT_L2_HR_Raster_C


--- Flood Progression Mode: Analyze WSE leading up to a flood event ---


Minimum longitude (e.g., -92.5):  -31.01342
Minimum latitude (e.g., 30.0):  -52.6557
Maximum longitude (e.g., -90.0):  -29.49418
Maximum latitude (e.g., 32.0):  -50.93321
Enter the flood event date (YYYY-MM-DD):  2024-05-06
How many days in advance do you want to look?  30
How many granules do you want to analyze? (e.g., 5):  2



--- Initializing for bounding box: (-31.01342, -52.6557, -29.49418, -50.93321) ---
Searching for SWOT_L2_HR_Raster_C granules in the specified area and timeframe...
Found 0 granules in total.
No valid granules found for the specified bounding box. Please check coordinates.
Cannot perform progression analysis without any granules.
